# 04C — Fresh proseg assignment

Run in the corresponding working environment. The pilot selects the first baseline bundle from the active manifest. After reviewing successful audits, choose remaining to process only new bundles. Use the identical bundle list for all three methods. Existing results are protected. Check seed coverage, count excess, status and fresh-run timing before comparing methods.

In [1]:
from pathlib import Path
import os, sys, json
import pandas as pd
from IPython.display import display
SUITE_DIR = next((p for parent in [Path.cwd(), *Path.cwd().parents]
                  for p in [parent, parent / "suite"] if (p / "benchmark_session.py").is_file()), None)
if SUITE_DIR is None:
    raise FileNotFoundError("Open this notebook from the unpacked public benchmark (or its suite/ directory).")
sys.path.insert(0, str(SUITE_DIR))
from benchmark_session import start
from p06212_benchmark.common import load_config, output_root, write_json, read_json
CONFIG_PATH, config, ROOT = start()

ACTIVE SPECIMEN: ovarian 
Configuration: /home/domino/reny28/Projects/ShortProject/Xenium_pseudo_VisiumHD01/public_prime_benchmark/ovarian/pilot_01/config.json 
Run directory: /home/domino/reny28/Projects/ShortProject/Xenium_pseudo_VisiumHD01/public_prime_benchmark/ovarian/pilot_01/results_root


In [2]:
from p06212_benchmark.runner import launch_method
from p06212_benchmark.methods import validate_input, environment_report
METHOD = "proseg"
METHOD_CONFIG = dict(config["methods"][METHOD])
display(pd.DataFrame([environment_report()]))
manifest = pd.read_csv(ROOT / "condition_manifest.csv", dtype={"bundle_id":str,"condition_id":str})
# Begin with one baseline bundle. After compatibility/count audits pass,
# choose the same bundle/condition list in all three method notebooks.
SCOPE = "all_fresh"  # pilot, remaining, or all_fresh. Each destination must be NEW.
all_ids = list(dict.fromkeys(manifest.loc[manifest.condition_id.eq("baseline"),"bundle_id"]))
if not all_ids:
    raise ValueError("No frozen baseline bundles in condition_manifest.csv; complete 03 first.")
if SCOPE not in {"pilot","remaining","all_fresh"}:
    raise ValueError("Unknown SCOPE")
BUNDLE_IDS = all_ids[:1] if SCOPE=="pilot" else all_ids[1:] if SCOPE=="remaining" else all_ids
CONDITIONS = ["baseline"]
REPEATS = 1  # Use repeated fresh runs (e.g. 3) for timing/stochastic summaries.
jobs = manifest.loc[manifest.bundle_id.isin(BUNDLE_IDS) & manifest.condition_id.isin(CONDITIONS)].copy()
display(jobs[["bundle_id","condition_id","input_dir"]])

,python,executable,packages,cpu_affinity
0,"3.13.7 (main, Sep 18 2025, 19:47:49) [Clang 20...",/home/domino/reny28/Projects/Visium_projects/T...,"{'numpy': '2.4.4', 'pandas': '2.3.3', 'scipy':...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."


,bundle_id,condition_id,input_dir
0,ovarian_B01,baseline,/home/domino/reny28/Projects/ShortProject/Xeni...
1,ovarian_B02,baseline,/home/domino/reny28/Projects/ShortProject/Xeni...
2,ovarian_B03,baseline,/home/domino/reny28/Projects/ShortProject/Xeni...
3,ovarian_B04,baseline,/home/domino/reny28/Projects/ShortProject/Xeni...
4,ovarian_B05,baseline,/home/domino/reny28/Projects/ShortProject/Xeni...
5,ovarian_B06,baseline,/home/domino/reny28/Projects/ShortProject/Xeni...
6,ovarian_B07,baseline,/home/domino/reny28/Projects/ShortProject/Xeni...
7,ovarian_B08,baseline,/home/domino/reny28/Projects/ShortProject/Xeni...


## Verify the installed native HD coordinate convention

The historical command fit a row/column affine and supplied each transform as **three separate CLI
arguments**. The new canvas has a different origin and scale. First run the asymmetric toy fixture
below against your actual executable. It tests the observed molecular coordinates and mask transform,
then saves evidence bound to that executable's version/hash. It does not score this study performance.

If the test fails, preserve its report and log. Inspect the installed version/axis convention before
changing `axis_order`; never remove the validation to make a failed orientation pass unnoticed.

In [5]:
from p06212_benchmark.methods import validate_proseg_coordinates
#PROSEG_EXE = Path(METHOD_CONFIG.get("executable") or "").expanduser()
PROSEG_EXE = Path("/home/domino/reny28/Projects/CosMX_projects/"
              "P06364_CRC_DS20250227_48371/code/Bash/.tools/bin/proseg").expanduser()
print("Absolute Proseg executable:", PROSEG_EXE)
if not PROSEG_EXE.is_absolute() or not PROSEG_EXE.is_file():
    raise FileNotFoundError(PROSEG_EXE)
proof_dir = ROOT / "compatibility" / "proseg_coordinates"
proof_path = proof_dir / "coordinate_validation.json"
if not proof_path.exists():
    validate_proseg_coordinates(str(PROSEG_EXE), proof_dir,
        axis_order=METHOD_CONFIG.get("axis_order","row_col"), image_mpp=config["design"]["image_mpp"])
METHOD_CONFIG["coordinate_validation_json"] = str(proof_path)
display(pd.DataFrame([read_json(proof_path)]))

Absolute Proseg executable: /home/domino/reny28/Projects/CosMX_projects/P06364_CRC_DS20250227_48371/code/Bash/.tools/bin/proseg


,status,executable,executable_sha256,version,proseg_axis_order,image_mpp,mask_pixel_center_offset,n_observed_molecules,n_masks,max_unchanged_mask_centroid_error_um,purpose
0,passed,/home/domino/reny28/Projects/CosMX_projects/P0...,4c72e1d07febb04be940a469e6b9c9befca5490c66e7e8...,proseg 3.1.1,row_col,0.5,0.5,5580,12,2.900363,native reader coordinates/counts and mask tran...


## Validate shared inputs and run

The adapters retain raw counts and stable seed IDs, record runtime versions/hashes, and audit missing
seeds and count excess/loss. A failed method is recorded as a failure; it must not silently disappear
from a comparison. Outputs go to a fresh repeat directory; existing results are never overwritten.
Close unrelated jobs and use the same CPU/GPU/storage conditions for comparable timings.

In [ ]:
if jobs.empty: raise ValueError("Scope selects no bundles. A one-bundle pilot is already all bundles.")
first = jobs.iloc[0]
_, _, _, _, diagnostics = validate_input(first.input_dir)
display(pd.DataFrame([diagnostics]))
method_name = METHOD if METHOD != "smurf" else "smurf_" + METHOD_CONFIG.get("variant","lite")
for row in jobs.itertuples():
    for repeat in range(1, REPEATS + 1):
        destination = ROOT / "results" / row.bundle_id / row.condition_id / method_name / f"repeat_{repeat:02d}"
        launch_method(METHOD, row.input_dir, destination, METHOD_CONFIG, python_executable=METHOD_CONFIG.get("python_executable"))

In [ ]:
for summary_path in sorted((ROOT / "results").glob(f"*/*/{method_name}/repeat_*/run_summary.json")):
    summary = read_json(summary_path)
    print(summary_path)
    display(pd.DataFrame([{key:summary.get(key) for key in [
        "status","n_shared_seeds","n_output_cells","input_counts","output_counts",
        "missing_output_seed_count","n_genes_with_excess_counts","total_runner_seconds",
        "peak_process_tree_rss_bytes"]}]))

## Before expanding the run

Check `gene_count_audit.csv`, `seed_audit.csv`, `run_summary.json` and the log. Confirm that the seed
hash is shared across methods for the same input condition. Native output cells may be dropped by a
method; the evaluator later counts them as zero recovery in the fixed baseline population.

For the main runtime comparison, report **total runner time** consistently. Proseg's native-process
stage includes its own I/O; it cannot be compared directly with the Python methods' assignment-only stage.
Sampled peak RAM is recorded when the operating system permits it; GPU memory is not measured by this
portable runner. Plotting and ResolVI/UCell are outside the primary assignment experiment.